# 01 — OCR extraction over your product photos (PackSure)

**Goal:** turn YOUR product images into reviewed OCR text drafts that you will annotate.

**What you do manually:**
1. Upload your product photos to `ml/dataset/raw/<product-group>/...` (see the cell below).
2. Run all cells — Tesseract extracts word-level text, boxes and confidence.
3. **READ every OCR draft and fix obvious OCR mistakes** — you are creating ground truth. Never train on text you have not read.
4. Annotate entities in notebook 02.

> OCR drafts are *not* training data until a human has reviewed them (`"reviewed": false` marks that).

In [ ]:
# Mount Google Drive (recommended) or upload a zip of ml/dataset/raw
from google.colab import drive
drive.mount('/content/drive')

# Expected layout (create it):
#   /content/drive/MyDrive/packsure/ml/dataset/raw/<product_id>/*.jpg
# Example: raw/maggi-noodles-70g/img1.jpg, img2.jpg ...
RAW_DIR = '/content/drive/MyDrive/packsure/ml/dataset/raw'
OUT_DIR = '/content/drive/MyDrive/packsure/ml/dataset/annotations/ocr_drafts'
print('RAW_DIR set to', RAW_DIR)

In [ ]:
# Install Tesseract + Python deps (Colab has neither by default)
!apt-get install -y -qq tesseract-ocr > /dev/null
!pip install -q pytesseract pillow

import shutil
print('tesseract:', shutil.which('tesseract'))

In [ ]:
# Batch OCR: image -> words (text, confidence, bbox) + assembled text
import sys, json
from pathlib import Path

# Clone the repo OR upload the ml/ folder next to this notebook.
# !git clone https://github.com/ShubhamKumar1729/packsure.git
REPO = Path('/content/packsure')  # adjust if you uploaded instead
sys.path.insert(0, str(REPO / 'ml' / 'scripts'))

from ocr_batch import run_directory, draft_annotation
from PIL import Image

raw = Path(RAW_DIR)
out = Path(OUT_DIR)
assert raw.exists(), f'{raw} not found - upload your photos first'

written = run_directory(raw, out)
print(f'OCR drafts written: {len(written)}')

In [ ]:
# Create annotation skeletons PRE-FILLED with suggested entity spans.
# The pattern extractor (the same one the app uses in degraded mode) proposes
# MRP / net quantity / dates / batch / contacts / party names automatically.
# You then VERIFY + CORRECT each suggestion in notebook 02 instead of drawing
# spans from scratch. Everything stays "reviewed": false until you check it.
import subprocess, sys
from pathlib import Path
drafts = sorted(Path(OUT_DIR).glob('*.ocr.json'))
print(f'OCR drafts: {len(drafts)}')
ann_dir = Path('/content/drive/MyDrive/packsure/ml/dataset/annotations/pending')
ann_dir.mkdir(parents=True, exist_ok=True)
result = subprocess.run(
    [sys.executable, str(REPO / 'ml' / 'scripts' / 'preannotate.py'), str(OUT_DIR), str(ann_dir)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('pre-annotation failed - see message above')
print('Pre-filled skeletons in', ann_dir, '- now REVIEW them (next cell).')

In [ ]:
# REVIEW pass: print each draft's OCR text so YOU can read and correct it.
# Fix text directly in the annotation JSON files (the `text` field) when OCR is wrong.
pending = sorted(ann_dir.glob('*.json'))
for path in pending[:20]:
    data = json.loads(path.read_text())
    print('=' * 60)
    print(path.name, f"(mean conf: check the .ocr.json for details)")
    print(data['text'] or '<OCR FOUND NO TEXT - replace this image or type the label text manually>')
print(f'\n{len(pending)} drafts to review. Next: notebook 02.')

**Checkpoint:** every image now has a human-reviewable OCR draft.
Do NOT continue to annotation until you have eyeballed every draft.